In [ ]:
# Example 1: public-API surface inventory.
import annotated_doc

public_names = [n for n in dir(annotated_doc) if not n.startswith('_')]
print(f"Public names ({len(public_names)}):")
for name in public_names:
    obj = getattr(annotated_doc, name)
    print(f"  {name}: {type(obj).__name__}")

# The primary export is `Doc` — used in typing.Annotated metadata.
assert hasattr(annotated_doc, 'Doc'), "annotated_doc must export Doc"


In [ ]:
# Example 2: construct a Doc instance + inspect its attributes.
from annotated_doc import Doc

d = Doc("The user's display name shown in the UI.")
print(f"Doc instance: {d!r}")
print(f"type: {type(d).__name__}")

# Doc typically stores the documentation string on a `documentation` attribute.
doc_text = getattr(d, 'documentation', None)
print(f"documentation attr: {doc_text!r}")
assert doc_text == "The user's display name shown in the UI."


In [ ]:
# Example 3: use Doc inside typing.Annotated on a function signature.
from typing import Annotated, get_type_hints

from annotated_doc import Doc

def greet(
    name: Annotated[str, Doc("Name of the person to greet.")],
    times: Annotated[int, Doc("How many times to repeat the greeting.")] = 1,
) -> Annotated[str, Doc("The composed greeting message.")]:
    return (f"Hello, {name}! " * times).strip()

result = greet("world", times=2)
print(result)
assert "Hello, world!" in result

hints = get_type_hints(greet, include_extras=True)
print(f"hints: {hints}")
assert 'name' in hints


In [ ]:
# Example 4: extract Doc metadata from an Annotated type via typing.get_args.
from typing import Annotated, get_args, get_type_hints

from annotated_doc import Doc

def add(
    x: Annotated[int, Doc("First addend.")],
    y: Annotated[int, Doc("Second addend.")],
) -> int:
    return x + y

hints = get_type_hints(add, include_extras=True)
for param, hint in hints.items():
    args = get_args(hint)
    doc_meta = next((a for a in args if isinstance(a, Doc)), None)
    if doc_meta is not None:
        print(f"{param}: {doc_meta.documentation}")

assert add(2, 3) == 5


In [ ]:
# Example 5: Doc equality + reuse across multiple parameters.
from typing import Annotated, get_args, get_type_hints

from annotated_doc import Doc

shared_doc = Doc("A non-negative integer count.")

def tally(
    apples: Annotated[int, shared_doc],
    oranges: Annotated[int, shared_doc],
) -> int:
    return apples + oranges

hints = get_type_hints(tally, include_extras=True)
docs_found = []
for hint in hints.values():
    for a in get_args(hint):
        if isinstance(a, Doc):
            docs_found.append(a)

print(f"Found {len(docs_found)} Doc instance(s) across the signature.")
print(f"First doc text: {docs_found[0].documentation!r}")

assert len(docs_found) >= 2
assert tally(3, 4) == 7
assert all(d.documentation == "A non-negative integer count." for d in docs_found)
